# ProstT5 Folding HMM T4 Evaluation

Clean start-to-end runner for the real CUDA/T4 benchmark.

This notebook uses only `prostT5/prostT5_benchmarks/folding_MSA` and writes resumable per-configuration results to `prostT5/results/folding_hmm_results/eval_result`. It runs:

- pure ProstT5 benchmark rows
- static HMM assisted decoding
- prefix HMM assisted decoding for `p = [1, 2, 4, 8, 12, 16]`
- `K = [1, 2, 4, 8, 12, 16]`
- homolog counts `N = [4, 8, 16, 32, 64]`

The benchmark records component runtimes, vRAM, runtime speedup, theoretical mean-k speedup, and acceptance rate. It refuses to run outside CUDA/T4 so CPU fallback results cannot be mixed in by accident.


In [1]:
#@title Runtime guard: require CUDA/T4 + mount Drive consistently
import os, sys, subprocess, importlib.util
from pathlib import Path

DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_MYDRIVE = DRIVE_MOUNT_POINT / 'MyDrive'

def _ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name])

_ensure_package('sentencepiece')
_ensure_package('accelerate')

# ProstT5 is known to work with Transformers 4.46.3 in this project.
try:
    import importlib.metadata as _metadata
    _tf_version = _metadata.version('transformers')
except Exception:
    _tf_version = None
if _tf_version != '4.46.3':
    print(f'Installing transformers==4.46.3 (current: {_tf_version}) ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.46.3'])


def running_in_colab() -> bool:
    return Path('/content').exists()


def ensure_colab_drive_mounted() -> None:
    """Mount Google Drive once, always at /content/drive.

    The rest of the notebook only searches under /content/drive/MyDrive, so we
    avoid the common Colab error where Drive is mounted or referenced through a
    different path.
    """
    if not running_in_colab():
        return
    if DRIVE_MYDRIVE.exists():
        print(f'Drive already mounted at {DRIVE_MOUNT_POINT}')
        return
    try:
        from google.colab import drive
    except Exception as exc:
        raise RuntimeError('Google Drive paths are configured, but google.colab.drive is unavailable.') from exc
    drive.mount(str(DRIVE_MOUNT_POINT), force_remount=False)
    if not DRIVE_MYDRIVE.exists():
        raise RuntimeError(f'Drive mount failed: {DRIVE_MYDRIVE} does not exist after mounting.')


ensure_colab_drive_mounted()

import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. In Colab choose Runtime > Change runtime type > T4 GPU, then run all cells again.')

GPU_NAME = torch.cuda.get_device_name(0)
REQUIRE_T4 = True
if REQUIRE_T4 and 'T4' not in GPU_NAME:
    raise RuntimeError(f'This notebook is configured for T4. Current GPU is {GPU_NAME!r}. Set REQUIRE_T4=False only if you intentionally use another CUDA GPU.')

torch.backends.cuda.matmul.allow_tf32 = True
print('torch:', torch.__version__)
print('GPU  :', GPU_NAME)
print('Drive:', DRIVE_MYDRIVE if DRIVE_MYDRIVE.exists() else 'not mounted / not Colab')


Drive already mounted at /content/drive
torch: 2.11.0+cu128
GPU  : Tesla T4
Drive: /content/drive/MyDrive


In [2]:
#@title Locate repo/data/script/output folder
from pathlib import Path
import os

NOTEBOOK_DIR = Path.cwd().resolve()
DATA_REL = Path('prostT5_benchmarks') / 'folding_MSA'
SCRIPT_NAME = 'folding_hmm_t4_eval.py'


def _candidate_bases(start: Path):
    bases = []
    for env_name in ('PROSTT5_REPO', 'PROST_DIR', 'PROSTT5_DIR'):
        if os.environ.get(env_name):
            bases.append(Path(os.environ[env_name]).expanduser())
    bases.extend([start, *start.parents])
    bases.extend([
        # Preferred Colab/Drive layout from the MSA factory run.
        Path('/content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5'),
        Path('/content/drive/MyDrive/Speculative-Decoding-ProstT5'),

        # Local Colab clone fallbacks.
        Path('/content/Speculative-Decoding-ProstT5/prostT5'),
        Path('/content/Speculative-Decoding-ProstT5'),

        # Other Drive layouts seen during local/Colab sync.
        Path('/content/drive/MyDrive'),
        Path('/content/drive/MyDrive/prostT5'),
        Path('/content/drive/MyDrive/TUM MS Informatics/courses/SoSe 26/PP 1/Speculative-Decoding-ProstT5/prostT5'),
        Path('/content/drive/MyDrive/TUM MS Informatics/courses/SoSe 26/PP 1/Speculative-Decoding-ProstT5'),
    ])
    seen = set()
    for base in bases:
        base = base.resolve() if base.exists() else base.absolute()
        if base in seen:
            continue
        seen.add(base)
        yield base


def _as_prost_dir(base: Path) -> Path:
    # Accept either the repository root (.../Speculative-Decoding-ProstT5) or
    # the prostT5 folder itself (.../Speculative-Decoding-ProstT5/prostT5).
    if (base / DATA_REL).exists() or (base / SCRIPT_NAME).exists():
        return base
    return base / 'prostT5'


PROST_DIR = None
found_data_dirs = []
found_script_dirs = []
for base in _candidate_bases(NOTEBOOK_DIR):
    cand = _as_prost_dir(base)
    data_ok = (cand / DATA_REL).exists()
    script_ok = (cand / SCRIPT_NAME).exists()
    if data_ok:
        found_data_dirs.append(cand)
    if script_ok:
        found_script_dirs.append(cand)
    if data_ok:
        PROST_DIR = cand
        break

if PROST_DIR is None:
    searched = '\n  '.join(str(_as_prost_dir(x)) for x in list(_candidate_bases(NOTEBOOK_DIR))[:40])
    msg = (
        'Could not find prostT5_benchmarks/folding_MSA.\n\n'
        f'Data folders found:\n  ' + ('\n  '.join(map(str, found_data_dirs)) if found_data_dirs else '(none)') + '\n\n'
        f'Script folders found:\n  ' + ('\n  '.join(map(str, found_script_dirs)) if found_script_dirs else '(none)') + '\n\n'
        f'Searched prost dirs:\n  {searched}\n\n'
        'Fix: use /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5 as the shared Drive prostT5 folder, with prostT5_benchmarks/folding_MSA inside it.'
    )
    raise FileNotFoundError(msg)

REPO_ROOT = PROST_DIR.parent if PROST_DIR.name == 'prostT5' else PROST_DIR
DATA_DIR = PROST_DIR / 'prostT5_benchmarks' / 'folding_MSA'
RESULT_DIR = PROST_DIR / 'results' / 'folding_hmm_results' / 'eval_result'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print('Prost dir :', PROST_DIR)
print('Repo root :', REPO_ROOT)
print('Data dir  :', DATA_DIR)
print('Eval script:', PROST_DIR / SCRIPT_NAME, '(will be bootstrapped if missing)')
print('Output dir:', RESULT_DIR)


Prost dir : /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5
Repo root : /content/drive/MyDrive/Speculative-Decoding-ProstT5
Data dir  : /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA
Eval script: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/folding_hmm_t4_eval.py (will be bootstrapped if missing)
Output dir: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/results/folding_hmm_results/eval_result


In [3]:
#@title Bootstrap eval helper scripts into the mounted prostT5 folder
# Colab can see the Drive data folder even when helper .py files have not been synced there.
# This cell writes the exact helper scripts needed by the runner into PROST_DIR.
from pathlib import Path

_PROFILE_SWEEP_SOURCE = '"""Offline drafter-quality sweep for the AA->3Di FOLDING direction.\n\nGoal: for the local `prostT5_benchmarks/folding_MSA` dataset, measure DRAFTER ACCURACY and MEAN K\nof the folding HMM/profile drafter\nacross:\n    * number of used homologs   N in {4, 8, 16, 32, 64}\n    * fixed draft length        K in {1, 2, 4, 8, 12, 16}\n    * prefix-context order      p in {static, 1, 2, 4, 8, 12, 16}   ("p" knob)\n\nWhy this runs with NO GPU\n-------------------------\nUnder *greedy* speculative decoding the generated sequence is provably identical\nto plain enc-dec greedy, for any drafter. So:\n    drafter accuracy = fraction of positions where the drafter\'s argmax 3Di token\n                       equals ProstT5\'s greedy 3Di token at that position;\n    mean k           = a deterministic simulation over that match pattern.\nBoth need only (a) the per-position drafter distribution and (b) ProstT5\'s greedy\n3Di output per protein (the "reference"). The drafter distribution is rebuilt from\n`homologs_projected_to_query_3di.fasta` — the dataset already folded every homolog\nwith ProstT5 and projected it to query columns, which is exactly what the team\'s\n`FamilyFoldingHMMDrafter._build_logits` does internally. The reference is taken\nfrom `query_3di.fasta` (which is itself ProstT5\'s greedy fold of the query).\n\nThis file replicates the team\'s drafter math from\n`prostT5_probabilistic_drafter_folding.ipynb` (HMM-folding branch):\n    HMM_SMOOTHING = 0.5, MAX_CONTEXT_P = 16, static counts + context_counts backoff,\n    PrefixAwareFoldingHMMDrafter(max_p=p).\n\nRun:\n    python3 folding_profile_sweep.py --data prostT5_benchmarks/folding_MSA \\\n        --out folding_profile_sweep_results\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport math\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n# --- team constants (prostT5_probabilistic_drafter_folding.ipynb) -------------\nHMM_SMOOTHING = 0.5\nMAX_CONTEXT_P = 16\nN_HOMOLOG_VALUES = [4, 8, 16, 32, 64]\nK_VALUES = [1, 2, 4, 8, 12, 16]\nP_VALUES = [1, 2, 4, 8, 12, 16]       # prefix-context orders; "static" added too\nGAP_CHARS = {"-", ".", "*"}\nSCRIPT_DIR = Path(__file__).resolve().parent\n\n\n# --- IO -----------------------------------------------------------------------\n\ndef read_fasta(path: Path) -> list[tuple[str, str]]:\n    recs: list[tuple[str, str]] = []\n    name, buf = None, []\n    for ln in path.read_text().splitlines():\n        if ln.startswith(">"):\n            if name is not None:\n                recs.append((name, "".join(buf)))\n            name, buf = ln[1:].split()[0] if ln[1:].split() else ln[1:], []\n        else:\n            buf.append(ln.strip())\n    if name is not None:\n        recs.append((name, "".join(buf)))\n    return recs\n\n\ndef load_protein(d: Path):\n    """Return (uid, query_3di_or_None, [projected_3di homolog strings])."""\n    uid = d.name\n    proj = [s for _, s in read_fasta(d / "homologs_projected_to_query_3di.fasta")]\n    q3_path = d / "query_3di.fasta"\n    q3 = read_fasta(q3_path)[0][1] if q3_path.exists() else None\n    return uid, q3, proj\n\n\n# --- drafter (faithful replica of FamilyFoldingHMMDrafter) --------------------\n\nclass FoldingProfileDrafter:\n    """Static column counts + prefix context_counts, built from N projected\n    homolog 3Di rows (already aligned to query columns)."""\n\n    def __init__(self, projected_rows: list[str], L: int, sym_to_idx: dict[str, int],\n                 smoothing: float = HMM_SMOOTHING, max_context_p: int = MAX_CONTEXT_P):\n        self.L = L\n        self.S = len(sym_to_idx)\n        self.sym_to_idx = sym_to_idx\n        self.smoothing = smoothing\n        self.max_context_p = max_context_p\n\n        counts = np.full((L, self.S), smoothing, dtype=np.float64)\n        self.context_counts: dict[tuple[int, tuple], np.ndarray] = defaultdict(\n            lambda: np.zeros(self.S, dtype=np.float64)\n        )\n\n        for row in projected_rows:\n            projected = list(row[:L]) + [None] * max(0, L - len(row))\n            projected = [c if c in sym_to_idx else None for c in projected]\n            observed = [(pos, c) for pos, c in enumerate(projected) if c is not None]\n            if not observed:\n                continue\n            for pos, tok in observed:\n                counts[pos, sym_to_idx[tok]] += 1.0\n            for pos, tok in observed:\n                for ctx_len in range(min(max_context_p, pos), -1, -1):\n                    ctx = projected[pos - ctx_len:pos]\n                    if any(c is None for c in ctx):\n                        continue\n                    self.context_counts[(pos, tuple(ctx))][sym_to_idx[tok]] += 1.0\n\n        self.static_counts = counts\n        self.static_argmax = counts.argmax(axis=1)   # (L,)\n        # drafter "confidence" = top-1 probability of the column distribution.\n        # This is what HF\'s assistant_confidence_threshold compares against.\n        self.static_conf = (counts.max(axis=1) / counts.sum(axis=1))  # (L,)\n\n    def pred_conf_for_prefix(self, prefix_syms: list, max_p: int) -> tuple[int, float]:\n        """(argmax class, top-1 prob) at position len(prefix), conditioned on up\n        to max_p preceding reference symbols (with backoff). The probability is\n        the drafter\'s confidence used by the dynamic-K policy."""\n        pos = len(prefix_syms)\n        if pos >= self.L:\n            return -1, 0.0\n        available = min(max_p, pos)\n        for ctx_len in range(available, -1, -1):\n            ctx = tuple(prefix_syms[-ctx_len:]) if ctx_len > 0 else tuple()\n            key = (pos, ctx)\n            if key in self.context_counts:\n                c = self.context_counts[key] + self.smoothing\n                return int(c.argmax()), float(c.max() / c.sum())\n        return int(self.static_argmax[pos]), float(self.static_conf[pos])\n\n    def argmax_for_prefix(self, prefix_syms: list, max_p: int) -> int:\n        """argmax 3Di class at position len(prefix), conditioned on up to max_p\n        preceding *reference* symbols (with backoff). Mirrors emission_for_prefix."""\n        pos = len(prefix_syms)\n        if pos >= self.L:\n            return -1\n        available = min(max_p, pos)\n        for ctx_len in range(available, -1, -1):\n            ctx = tuple(prefix_syms[-ctx_len:]) if ctx_len > 0 else tuple()\n            key = (pos, ctx)\n            if key in self.context_counts:\n                return int(self.context_counts[key].argmax())\n        return int(self.static_argmax[pos])\n\n\n# --- greedy spec-decode simulation over a match mask --------------------------\n\ndef simulate_fixed_k(match: list[bool], K: int):\n    """Greedy Leviathan accounting: each step proposes K drafts, accepts the\n    leading run of matches, then emits one verifier token (+1). Returns\n    (n_steps, mean_accepted, mean_tokens_per_step, acceptance_rate)."""\n    L = len(match)\n    cursor = steps = accepted_total = proposed_total = 0\n    while cursor < L:\n        k = min(K, L - cursor)\n        a = 0\n        while a < k and match[cursor + a]:\n            a += 1\n        accepted_total += a\n        proposed_total += k\n        steps += 1\n        cursor += a + 1          # a accepted drafts + 1 verifier token (corr/bonus)\n    mean_accepted = accepted_total / steps\n    mean_tokens_per_step = L / steps          # tokens generated per model call == "mean k"\n    acc_rate = accepted_total / proposed_total if proposed_total else 0.0\n    return steps, mean_accepted, mean_tokens_per_step, acc_rate\n\n\n# --- main sweep ---------------------------------------------------------------\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default=str(SCRIPT_DIR / "prostT5_benchmarks" / "folding_MSA"),\n                    help="path to the folding_MSA/ directory")\n    ap.add_argument("--out", default="folding_profile_sweep_results")\n    ap.add_argument("--refs", default=None,\n                    help="optional refs_3di.json (uid -> ProstT5 greedy 3Di). "\n                         "Overrides query_3di and covers proteins missing it.")\n    args = ap.parse_args()\n\n    import json\n    refs_json = json.loads(Path(args.refs).read_text()) if args.refs else {}\n\n    data = Path(args.data).resolve()\n    out = Path(args.out).resolve()\n    out.mkdir(parents=True, exist_ok=True)\n\n    all_dirs = sorted(p for p in data.iterdir() if p.is_dir())\n    dirs = [p for p in all_dirs if (p / "homologs_projected_to_query_3di.fasta").exists()]\n    skipped_incomplete = [p.name for p in all_dirs if p not in dirs]\n    print(f"Loading {len(dirs)} proteins from {data}")\n    if skipped_incomplete:\n        print("Skipped incomplete protein directories: " + ", ".join(skipped_incomplete))\n\n    # 3Di alphabet = sorted unique non-gap symbols across all projected rows\n    alpha = set()\n    loaded = []\n    for d in dirs:\n        uid, q3, proj = load_protein(d)\n        loaded.append((uid, q3, proj))\n        for row in proj:\n            alpha |= {c for c in row if c not in GAP_CHARS}\n    THREEDI = "".join(sorted(alpha))\n    sym_to_idx = {c: i for i, c in enumerate(THREEDI)}\n    print(f"3Di alphabet ({len(THREEDI)}): {THREEDI}")\n\n    acc_rows = []     # per (protein, N, mode, p): drafter accuracy\n    k_rows = []       # per (protein, N, mode, p, K): mean k etc.\n    n_scored = 0\n    n_noref = []\n\n    for uid, q3, proj in loaded:\n        ref_str = refs_json.get(uid) or q3\n        ref_source = "refs_json" if (uid in refs_json) else "query_3di"\n        if ref_str is None:\n            n_noref.append(uid)\n            continue\n        L = len(ref_str)\n        ref = list(ref_str)\n        n_scored += 1\n\n        for N in N_HOMOLOG_VALUES:\n            rows_N = proj[:N]\n            n_avail = len(rows_N)\n            if len(proj) < N:\n                print(f\'protein "{uid}" has only {len(proj)} homologs, requires {N}; using {n_avail}.\')\n            drafter = FoldingProfileDrafter(rows_N, L, sym_to_idx)\n\n            # ----- static (prefix-blind) -----\n            static_pred = drafter.static_argmax\n            ref_idx = np.array([sym_to_idx.get(c, -1) for c in ref])\n            match_static = [bool(static_pred[j] == ref_idx[j] and ref_idx[j] >= 0)\n                            for j in range(L)]\n            acc_static = float(np.mean(match_static))\n            acc_rows.append(dict(protein_id=uid, length=L, n_homologs=N,\n                                 n_homologs_used=n_avail, n_homologs_avail=len(proj),\n                                 mode="static", p="",\n                                 drafter_accuracy=round(acc_static, 4),\n                                 ref_source=ref_source))\n            for K in K_VALUES:\n                st, ma, mt, ar = simulate_fixed_k(match_static, K)\n                k_rows.append(dict(protein_id=uid, length=L, n_homologs=N, mode="static",\n                                   n_homologs_used=n_avail, n_homologs_avail=len(proj),\n                                   p="", K=K, drafter_accuracy=round(acc_static, 4),\n                                   mean_accepted=round(ma, 4),\n                                   mean_tokens_per_step=round(mt, 4),\n                                   acceptance_rate=round(ar, 4), n_steps=st,\n                                   ref_source=ref_source))\n\n            # ----- prefix-aware p values -----\n            for p in P_VALUES:\n                match_p = []\n                for j in range(L):\n                    pred = drafter.argmax_for_prefix(ref[:j], p)\n                    match_p.append(bool(pred == ref_idx[j] and ref_idx[j] >= 0))\n                acc_p = float(np.mean(match_p))\n                acc_rows.append(dict(protein_id=uid, length=L, n_homologs=N,\n                                     n_homologs_used=n_avail, n_homologs_avail=len(proj),\n                                     mode="prefix", p=p,\n                                     drafter_accuracy=round(acc_p, 4),\n                                     ref_source=ref_source))\n                for K in K_VALUES:\n                    st, ma, mt, ar = simulate_fixed_k(match_p, K)\n                    k_rows.append(dict(protein_id=uid, length=L, n_homologs=N,\n                                       n_homologs_used=n_avail, n_homologs_avail=len(proj),\n                                       mode="prefix", p=p, K=K,\n                                       drafter_accuracy=round(acc_p, 4),\n                                       mean_accepted=round(ma, 4),\n                                       mean_tokens_per_step=round(mt, 4),\n                                       acceptance_rate=round(ar, 4), n_steps=st,\n                                   ref_source=ref_source))\n\n    # write per-protein CSVs\n    _write_csv(out / "drafter_accuracy.csv", acc_rows)\n    _write_csv(out / "mean_k_sweep.csv", k_rows)\n    print(f"\\nScored {n_scored} proteins (reference = query_3di greedy fold).")\n    if n_noref:\n        print(f"Skipped (no query_3di): {\', \'.join(n_noref)}")\n\n    # ----- aggregate across proteins -----\n    agg = _aggregate(k_rows)\n    _write_csv(out / "summary_by_config.csv", agg)\n    _print_headline(agg)\n\n    print(f"\\nWrote:\\n  {out/\'drafter_accuracy.csv\'}\\n  {out/\'mean_k_sweep.csv\'}"\n          f"\\n  {out/\'summary_by_config.csv\'}")\n    return 0\n\n\ndef _aggregate(k_rows):\n    groups = defaultdict(list)\n    for r in k_rows:\n        groups[(r["n_homologs"], r["mode"], r["p"], r["K"])].append(r)\n    agg = []\n    for (N, mode, p, K), rs in sorted(groups.items(), key=lambda kv: (kv[0][0], str(kv[0][1]), str(kv[0][2]), kv[0][3])):\n        agg.append(dict(\n            n_homologs=N, mode=mode, p=p, K=K, n_proteins=len(rs),\n            mean_drafter_accuracy=round(float(np.mean([r["drafter_accuracy"] for r in rs])), 4),\n            mean_mean_k=round(float(np.mean([r["mean_tokens_per_step"] for r in rs])), 4),\n            median_mean_k=round(float(np.median([r["mean_tokens_per_step"] for r in rs])), 4),\n            mean_acceptance_rate=round(float(np.mean([r["acceptance_rate"] for r in rs])), 4),\n        ))\n    return agg\n\n\ndef _print_headline(agg):\n    print("\\n=== Drafter accuracy vs #homologs (static, K-independent) ===")\n    seen = set()\n    for r in agg:\n        if r["mode"] == "static" and r["n_homologs"] not in seen:\n            seen.add(r["n_homologs"])\n            print(f"  N={r[\'n_homologs\']:>2}: accuracy={r[\'mean_drafter_accuracy\']:.3f}")\n    print("\\n=== Best mean-k config (mean tokens per model call, higher=faster) ===")\n    best = max(agg, key=lambda r: r["mean_mean_k"])\n    print(f"  N={best[\'n_homologs\']} mode={best[\'mode\']} p={best[\'p\']} K={best[\'K\']}"\n          f"  -> mean_k={best[\'mean_mean_k\']:.2f}  accuracy={best[\'mean_drafter_accuracy\']:.3f}")\n    headline_k = 4\n    print(f"\\n=== mean_k at K={headline_k}, N=64, static vs prefix-p ===")\n    for r in agg:\n        if r["n_homologs"] == 64 and r["K"] == headline_k:\n            tag = "static" if r["mode"] == "static" else f"p={r[\'p\']}"\n            print(f"  {tag:>8}: mean_k={r[\'mean_mean_k\']:.2f}  acc={r[\'mean_drafter_accuracy\']:.3f}")\n\n\ndef _write_csv(path: Path, rows: list[dict]):\n    if not rows:\n        path.write_text("")\n        return\n    keys = list(rows[0].keys())\n    with open(path, "w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=keys)\n        w.writeheader()\n        w.writerows(rows)\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
_FOLDING_HMM_T4_EVAL_SOURCE = '"""CUDA/T4 folding-HMM benchmark over prostT5/prostT5_benchmarks/folding_MSA.\n\nThis is the real runtime pipeline. It refuses to run without CUDA, uses only the\nlocal prostT5_benchmarks/folding_MSA directory, and writes resumable per-protein rows under\nresults/folding_hmm_results/eval_result.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport functools\nimport gc\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom transformers import AutoModelForSeq2SeqLM, GenerationConfig, PreTrainedModel, T5Config, T5Tokenizer\nfrom transformers.generation.utils import GenerationMixin\nfrom transformers.modeling_outputs import BaseModelOutput, Seq2SeqLMOutput\n\nfrom folding_profile_sweep import (\n    FoldingProfileDrafter,\n    GAP_CHARS,\n    K_VALUES,\n    N_HOMOLOG_VALUES,\n    P_VALUES,\n    load_protein,\n    read_fasta,\n    simulate_fixed_k,\n)\n\n\nSCRIPT_DIR = Path(__file__).resolve().parent\nDEFAULT_DATA = SCRIPT_DIR / "prostT5_benchmarks" / "folding_MSA"\nDEFAULT_OUT = SCRIPT_DIR / "results" / "folding_hmm_results" / "eval_result"\nEXECUTION_MODE = "t4_cuda_prostt5"\nMODEL_NAME = "Rostlab/ProstT5_fp16"\nCOMPLETION_FIELDS = ("protein_id", "mode", "p", "K", "n_homologs_requested", "execution_mode")\nCONFIGS_PER_N = len(K_VALUES) * (1 + len(P_VALUES))\nprint = functools.partial(print, flush=True)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", default=str(DEFAULT_DATA))\n    parser.add_argument("--out", default=str(DEFAULT_OUT))\n    parser.add_argument("--filename", default="folding_hmm_eval_result.csv")\n    parser.add_argument("--benchmark-fasta", default=None,\n                        help="optional FASTA whose record IDs define the proteins to evaluate, e.g. test_set_AA_2.fasta")\n    parser.add_argument("--model-name", default=MODEL_NAME)\n    parser.add_argument("--protein-limit", type=int, default=None)\n    parser.add_argument("--max-configs", type=int, default=None,\n                        help="optional smoke-test cap on new non-benchmark configs")\n    parser.add_argument("--progress-every", type=int, default=10,\n                        help="print a text progress line after this many new configs")\n    parser.add_argument("--k-values", default=None,\n                        help="comma-separated K values; default uses the full grid")\n    parser.add_argument("--p-values", default=None,\n                        help="comma-separated prefix p values; default uses the full grid; use none for static-only")\n    parser.add_argument("--n-values", default=None,\n                        help="comma-separated homolog-count values; default uses the full grid")\n    parser.add_argument("--min-homologs", type=int, default=0,\n                        help="exclude proteins with fewer projected homolog rows before running configs")\n    parser.add_argument("--include-static", action="store_true",\n                        help="also run static HMM rows for the selected K/N values")\n    parser.add_argument("--skip-baseline", action="store_true",\n                        help="do not run pure ProstT5 baseline rows; speedup_runtime will be blank for new assisted rows")\n    parser.add_argument("--force", action="store_true")\n    args = parser.parse_args()\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("This benchmark must run on a CUDA GPU, e.g. Colab T4. Current torch.cuda.is_available() is False.")\n    device = torch.device("cuda:0")\n    torch.backends.cuda.matmul.allow_tf32 = True\n\n    data = Path(args.data).resolve()\n    out_dir = Path(args.out).resolve()\n    out_dir.mkdir(parents=True, exist_ok=True)\n    out_path = out_dir / args.filename\n    completed_path = out_dir / "folding_hmm_eval_completed_keys.csv"\n    summary_path = out_dir / "folding_hmm_eval_run_summary.csv"\n\n    existing_rows = [] if args.force else [\n        _normalize_row(r) for r in _read_rows(out_path)\n        if r.get("execution_mode") == EXECUTION_MODE\n    ]\n    completed = {_completion_key(r) for r in existing_rows if _has_key(r)}\n    print(f"Resume: {len(existing_rows)} existing T4 rows, {len(completed)} completed keys")\n\n    valid_dirs = sorted(\n        p for p in data.iterdir()\n        if p.is_dir() and (p / "homologs_projected_to_query_3di.fasta").exists()\n    )\n    requested_ids = _read_id_filter(args.benchmark_fasta)\n    if requested_ids is not None:\n        before = len(valid_dirs)\n        valid_dirs = [p for p in valid_dirs if p.name in requested_ids]\n        missing_ids = sorted(requested_ids - {p.name for p in valid_dirs})\n        print(\n            f"Benchmark FASTA filter: {len(valid_dirs)}/{before} MSA folders match "\n            f"{args.benchmark_fasta}"\n        )\n        if missing_ids:\n            print("Missing MSA folders for benchmark IDs: " + ", ".join(missing_ids[:30]))\n            if len(missing_ids) > 30:\n                print(f"... plus {len(missing_ids) - 30} more missing IDs")\n    if args.min_homologs:\n        before = len(valid_dirs)\n        valid_dirs = [\n            p for p in valid_dirs\n            if _count_fasta_records(p / "homologs_projected_to_query_3di.fasta") >= args.min_homologs\n        ]\n        print(\n            f"Min-homolog filter: {len(valid_dirs)}/{before} MSA folders have "\n            f">= {args.min_homologs} projected homologs"\n        )\n    loaded = [load_protein(d) for d in valid_dirs]\n    if args.protein_limit is not None:\n        loaded = loaded[:args.protein_limit]\n    k_values = _parse_int_list(args.k_values, K_VALUES)\n    p_values = _parse_int_list(args.p_values, P_VALUES)\n    n_values = _parse_int_list(args.n_values, N_HOMOLOG_VALUES)\n\n    sym_to_idx = _build_alphabet(loaded)\n    tokenizer = T5Tokenizer.from_pretrained(args.model_name, do_lower_case=False, legacy=True)\n    model = AutoModelForSeq2SeqLM.from_pretrained(\n        args.model_name,\n        low_cpu_mem_usage=True,\n        torch_dtype=torch.float16,\n    ).to(device).eval()\n    model = model.half()\n\n    three_di_alphabet = "".join(sym_to_idx.keys())\n    three_di_token_ids = [tokenizer.encode(f" {tok}", add_special_tokens=False)[0] for tok in three_di_alphabet]\n    token_id_to_3di = {tid: tok for tok, tid in zip(three_di_alphabet, three_di_token_ids)}\n    assistant = FoldingProfileAssistant(\n        model.config,\n        model.get_encoder(),\n        three_di_token_ids,\n        token_id_to_3di,\n        device,\n    ).to(device).eval()\n\n    new_rows: list[dict] = []\n    skipped_no_ref: list[str] = []\n    short_homologs: list[str] = []\n    new_configs = 0\n    total_remaining = _count_remaining_configs(\n        loaded, data, completed, k_values, p_values, n_values, args.include_static\n    )\n    if args.max_configs is not None:\n        total_remaining = min(total_remaining, args.max_configs)\n    selected_configs_per_n = len(k_values) * (len(p_values) + (1 if args.include_static else 0))\n    print(\n        f"Benchmark plan: {len(loaded)} proteins, {selected_configs_per_n} selected configs per N, "\n        f"selected K={k_values}, p={p_values}, N={n_values}, include_static={args.include_static}, "\n        f"remaining assisted configs={total_remaining}"\n    )\n\n    progress = _make_progress(total_remaining, desc="T4 folding HMM configs")\n    started_at = time.perf_counter()\n\n    for protein_i, (uid, q3, projected_rows) in enumerate(loaded, start=1):\n        aa_path = data / uid / "query_aa.fasta"\n        if q3 is None or not aa_path.exists():\n            skipped_no_ref.append(uid)\n            continue\n        aa = read_fasta(aa_path)[0][1].upper()\n        length = len(aa)\n        ref = list(q3)\n        ref_idx = np.array([sym_to_idx.get(c, -1) for c in ref])\n        homologs_avail = len(projected_rows)\n        protein_new_start = new_configs\n        print(f"\\n[{protein_i}/{len(loaded)}] {uid} L={length} homologs={homologs_avail}")\n\n        bench_key = _make_key(uid, "benchmark", "", "", "")\n        baseline = None\n        if not args.skip_baseline and bench_key not in completed:\n            print(f"  {uid}: running pure ProstT5 baseline")\n            baseline = run_generation(model, tokenizer, aa, device, assistant=None)\n            new_rows.append(_row(\n                uid=uid, protein_i=protein_i, length=length,\n                pipeline="prostT5_benchmark", mode="benchmark", p="", k="",\n                n_requested="", n_used="", n_avail=homologs_avail,\n                timings=baseline, seq=baseline["seq"], ref=q3,\n                mean_k=1.0, acceptance_rate=1.0, mean_accepted=0.0,\n                n_steps=length, drafter_accuracy="", speedup_runtime=1.0,\n            ))\n            _flush_checkpoints(out_path, completed_path, existing_rows, new_rows)\n        else:\n            baseline = None\n\n        baseline_wall = _existing_baseline_wall(existing_rows, uid)\n        if baseline_wall is None and baseline is not None:\n            baseline_wall = baseline["wall_s"]\n        if baseline_wall is None and not args.skip_baseline:\n            print(f"  {uid}: baseline already completed but wall_s was not found; runtime speedup will be blank for new rows")\n\n        for n_requested in n_values:\n            rows_n = projected_rows[:n_requested]\n            n_used = len(rows_n)\n            if homologs_avail < n_requested:\n                msg = f\'protein "{uid}" has only {homologs_avail} homologs, requires {n_requested}; using {n_used}.\'\n                print(msg)\n                short_homologs.append(msg)\n\n            build_t0 = time.perf_counter()\n            drafter = FoldingProfileDrafter(rows_n, len(ref), sym_to_idx)\n            drafter_build_s = time.perf_counter() - build_t0\n\n            configs = []\n            if args.include_static:\n                configs.extend(("static", "", k) for k in k_values)\n            configs.extend(("prefix", p, k) for p in p_values for k in k_values)\n            for mode, p_val, k in configs:\n                key = _make_key(uid, mode, p_val, k, n_requested)\n                if key in completed:\n                    continue\n                if args.max_configs is not None and new_configs >= args.max_configs:\n                    break\n\n                config_t0 = time.perf_counter()\n                match, drafter_accuracy = _match_mask(drafter, ref, ref_idx, mode, p_val)\n                steps, mean_accepted, mean_k, acceptance_rate = simulate_fixed_k(match, int(k))\n\n                assistant.set_active(drafter, mode=mode, p=int(p_val) if p_val != "" else 0)\n                assistant.generation_config.num_assistant_tokens = float(k)\n                try:\n                    timings = run_generation(model, tokenizer, aa, device, assistant=assistant)\n                    speedup_runtime = baseline_wall / timings["wall_s"] if baseline_wall and timings["wall_s"] > 0 else ""\n                    seq = timings["seq"]\n                    eval_status = "ok"\n                    error_message = ""\n                except torch.OutOfMemoryError as exc:\n                    print(f"  {uid}: CUDA OOM for mode={mode} p={p_val} K={k} N={n_requested}; recording oom row and continuing")\n                    _clear_cuda()\n                    timings = _empty_timings()\n                    speedup_runtime = ""\n                    seq = ""\n                    eval_status = "oom"\n                    error_message = _short_error(exc)\n                new_rows.append(_row(\n                    uid=uid, protein_i=protein_i, length=length,\n                    pipeline="folding_hmm_assisted", mode=mode, p=p_val, k=k,\n                    n_requested=n_requested, n_used=n_used, n_avail=homologs_avail,\n                    timings=timings, seq=seq, ref=q3,\n                    mean_k=mean_k, acceptance_rate=acceptance_rate,\n                    mean_accepted=mean_accepted, n_steps=steps,\n                    drafter_accuracy=drafter_accuracy,\n                    speedup_runtime=speedup_runtime,\n                    drafter_build_s=drafter_build_s,\n                    eval_status=eval_status,\n                    error_message=error_message,\n                ))\n                _flush_checkpoints(out_path, completed_path, existing_rows, new_rows)\n                new_configs += 1\n                elapsed_config = time.perf_counter() - config_t0\n                progress.update(1)\n                if args.progress_every and (new_configs % args.progress_every == 0):\n                    elapsed_total = time.perf_counter() - started_at\n                    wall_text = f"{timings[\'wall_s\']:.2f}s" if timings["wall_s"] != "" else eval_status\n                    print(\n                        f"  progress: new_configs={new_configs}/{total_remaining} "\n                        f"last={uid} mode={mode} p={p_val} K={k} N={n_requested} "\n                        f"wall={wall_text} cfg_elapsed={elapsed_config:.2f}s "\n                        f"elapsed={elapsed_total/60:.1f}min"\n                    )\n            if args.max_configs is not None and new_configs >= args.max_configs:\n                break\n        if args.max_configs is not None and new_configs >= args.max_configs:\n            break\n        print(f"  {uid}: added {new_configs - protein_new_start} assisted configs this run")\n\n    progress.close()\n\n    all_rows = _dedupe(existing_rows + new_rows)\n    _write_rows(out_path, all_rows, FIELDNAMES)\n    _write_rows(completed_path, [dict(zip(COMPLETION_FIELDS, _completion_key(r))) for r in all_rows], list(COMPLETION_FIELDS))\n    _write_rows(summary_path, [{\n        "data_dir": str(data),\n        "benchmark_fasta": str(Path(args.benchmark_fasta).resolve()) if args.benchmark_fasta else "",\n        "output_csv": str(out_path),\n        "completed_keys_csv": str(completed_path),\n        "proteins_found": len(valid_dirs),\n        "proteins_seen_this_run": len(loaded),\n        "proteins_scored_total": len({r["protein_id"] for r in all_rows}),\n        "rows_total": len(all_rows),\n        "rows_reused": len(existing_rows),\n        "rows_added": len(new_rows),\n        "force_recompute": args.force,\n        "skipped_no_query_or_aa": ";".join(skipped_no_ref),\n        "short_homolog_messages": len(short_homologs),\n        "execution_mode": EXECUTION_MODE,\n        "device": torch.cuda.get_device_name(0),\n    }])\n    print(f"Added {len(new_rows)} rows; total T4 rows now {len(all_rows)}")\n    print(f"Wrote {out_path}")\n    return 0\n\n\ndef _make_progress(total: int, desc: str):\n    try:\n        from tqdm.auto import tqdm\n        return tqdm(total=total, desc=desc, unit="cfg", dynamic_ncols=True)\n    except Exception:\n        return _TextProgress(total, desc)\n\n\nclass _TextProgress:\n    def __init__(self, total: int, desc: str):\n        self.total = total\n        self.desc = desc\n        self.n = 0\n        print(f"{desc}: 0/{total}")\n\n    def update(self, step: int = 1):\n        self.n += step\n\n    def close(self):\n        print(f"{self.desc}: {self.n}/{self.total} complete")\n\n\ndef _count_remaining_configs(\n    loaded,\n    data: Path,\n    completed: set[tuple[str, str, str, str, str, str]],\n    k_values: list[int],\n    p_values: list[int],\n    n_values: list[int],\n    include_static: bool,\n) -> int:\n    total = 0\n    for uid, q3, _projected_rows in loaded:\n        aa_path = data / uid / "query_aa.fasta"\n        if q3 is None or not aa_path.exists():\n            continue\n        for n_requested in n_values:\n            if include_static:\n                for k in k_values:\n                    if _make_key(uid, "static", "", k, n_requested) not in completed:\n                        total += 1\n            for p in p_values:\n                for k in k_values:\n                    if _make_key(uid, "prefix", p, k, n_requested) not in completed:\n                        total += 1\n    return total\n\n\ndef _parse_int_list(value: str | None, default: list[int]) -> list[int]:\n    if value is None or str(value).strip() == "":\n        return list(default)\n    if str(value).strip().lower() in {"none", "static-only", "static_only"}:\n        return []\n    return [int(part.strip()) for part in str(value).split(",") if part.strip()]\n\n\ndef _read_id_filter(fasta_path: str | None) -> set[str] | None:\n    if fasta_path is None or str(fasta_path).strip() == "":\n        return None\n    path = Path(fasta_path).resolve()\n    if not path.exists():\n        raise FileNotFoundError(f"benchmark FASTA not found: {path}")\n    ids = {name.split()[0].lstrip(">").strip() for name, _seq in read_fasta(path)}\n    if not ids:\n        raise ValueError(f"benchmark FASTA has no records: {path}")\n    return ids\n\n\ndef _count_fasta_records(path: Path) -> int:\n    if not path.exists():\n        return 0\n    return sum(1 for line in path.read_text().splitlines() if line.startswith(">"))\n\n\ndef _empty_timings():\n    return {\n        "encoder_s": "",\n        "drafter_s": "",\n        "drafter_forward_calls": "",\n        "decoder_s": "",\n        "wall_s": "",\n        "peak_vram_gb": "",\n        "seq": "",\n    }\n\n\nclass FoldingProfileAssistant(PreTrainedModel, GenerationMixin):\n    config_class = T5Config\n\n    def __init__(self, config, encoder, three_di_token_ids, token_id_to_3di, device):\n        super().__init__(config)\n        self._encoder = encoder\n        self._di_ids = torch.tensor(three_di_token_ids, device=device, dtype=torch.long)\n        self._token_id_to_3di = token_id_to_3di\n        self._device = device\n        self.config.is_encoder_decoder = True\n        self.config.decoder_start_token_id = config.decoder_start_token_id\n        self.generation_config = GenerationConfig(\n            num_assistant_tokens=5,\n            num_assistant_tokens_schedule="constant",\n            do_sample=False,\n            max_length=3000,\n        )\n        self._active = None\n        self._mode = "static"\n        self._p = 0\n        self.forward_s = 0.0\n        self.forward_calls = 0\n\n    def set_active(self, drafter, *, mode: str, p: int):\n        self._active = drafter\n        self._mode = mode\n        self._p = p\n        self.forward_s = 0.0\n        self.forward_calls = 0\n\n    def get_encoder(self):\n        return self._encoder\n\n    def _validate_model_kwargs(self, model_kwargs):\n        return\n\n    def _prepare_encoder_decoder_kwargs_for_generation(\n        self, inputs_tensor, model_kwargs, model_input_name, generation_config\n    ):\n        # The HMM/profile assistant does not encode the AA prompt. HuggingFace\'s\n        # default encoder-decoder assistant path tries to run a T5 encoder over\n        # decoder token ids, which can trigger CUDA index errors. A tiny dummy\n        # encoder output is enough because forward() ignores encoder_outputs.\n        if "encoder_outputs" not in model_kwargs:\n            batch = inputs_tensor.shape[0] if inputs_tensor is not None else 1\n            hidden = getattr(self.config, "d_model", 1024)\n            model_kwargs["encoder_outputs"] = BaseModelOutput(\n                last_hidden_state=torch.zeros(batch, 1, hidden, device=self._device, dtype=torch.float16)\n            )\n        return model_kwargs\n\n    def prepare_inputs_for_generation(self, decoder_input_ids, encoder_outputs=None, **kwargs):\n        return {"decoder_input_ids": decoder_input_ids, "encoder_outputs": encoder_outputs}\n\n    def forward(self, decoder_input_ids=None, encoder_outputs=None, **kwargs):\n        t0 = time.perf_counter()\n        seq_len = decoder_input_ids.shape[1]\n        vocab = self.config.vocab_size\n        logits = torch.full((1, seq_len, vocab), -1e4, device=decoder_input_ids.device)\n        if self._active is not None:\n            decoded = decoder_input_ids[0, 1:].tolist()\n            for pos in range(seq_len):\n                if self._mode == "prefix":\n                    prefix = [self._token_id_to_3di[t] for t in decoded[:pos] if t in self._token_id_to_3di]\n                    row = _prefix_log_probs(self._active, prefix, self._p)\n                else:\n                    row = _static_log_probs(self._active, pos)\n                row_t = torch.from_numpy(row).to(self._device, dtype=logits.dtype)\n                logits[0, pos, self._di_ids] = row_t\n        if self._device.type == "cuda":\n            torch.cuda.synchronize()\n        self.forward_s += time.perf_counter() - t0\n        self.forward_calls += 1\n        return Seq2SeqLMOutput(logits=logits)\n\n\ndef run_generation(model, tokenizer, aa: str, device, assistant: FoldingProfileAssistant | None):\n    _clear_cuda()\n    encoded = tokenizer([_format_aa(aa)], add_special_tokens=True, return_tensors="pt").to(device)\n    torch.cuda.reset_peak_memory_stats(device)\n    torch.cuda.synchronize()\n\n    with torch.inference_mode():\n        t0 = time.perf_counter()\n        encoder_outputs = model.get_encoder()(input_ids=encoded.input_ids, attention_mask=encoded.attention_mask)\n        torch.cuda.synchronize()\n        encoder_s = time.perf_counter() - t0\n\n        if assistant is not None:\n            assistant.forward_s = 0.0\n            assistant.forward_calls = 0\n        gen_kwargs = dict(\n            attention_mask=encoded.attention_mask,\n            encoder_outputs=encoder_outputs,\n            max_length=len(aa) + 2,\n            do_sample=False,\n            num_beams=1,\n            use_cache=True,\n        )\n        if assistant is not None:\n            gen_kwargs["assistant_model"] = assistant\n\n        t0 = time.perf_counter()\n        output = model.generate(**gen_kwargs)\n        torch.cuda.synchronize()\n        decoder_total_s = time.perf_counter() - t0\n        drafter_s = assistant.forward_s if assistant is not None else 0.0\n        wall_s = encoder_s + decoder_total_s\n        seq = _decode_3di(tokenizer, output[0])[:len(aa)]\n        peak_vram_gb = torch.cuda.max_memory_allocated(device) / 1e9\n\n    del encoded, encoder_outputs, output\n    gc.collect()\n    _clear_cuda()\n    return {\n        "encoder_s": encoder_s,\n        "drafter_s": drafter_s,\n        "drafter_forward_calls": assistant.forward_calls if assistant is not None else 0,\n        "decoder_s": max(decoder_total_s - drafter_s, 0.0),\n        "wall_s": wall_s,\n        "peak_vram_gb": peak_vram_gb,\n        "seq": seq,\n    }\n\n\ndef _match_mask(drafter, ref, ref_idx, mode, p_val):\n    if mode == "static":\n        match = [bool(drafter.static_argmax[j] == ref_idx[j] and ref_idx[j] >= 0) for j in range(len(ref))]\n    else:\n        p = int(p_val)\n        match = [bool(drafter.argmax_for_prefix(ref[:j], p) == ref_idx[j] and ref_idx[j] >= 0) for j in range(len(ref))]\n    return match, float(np.mean(match))\n\n\ndef _static_log_probs(drafter, pos: int) -> np.ndarray:\n    pos = min(pos, drafter.L - 1)\n    counts = drafter.static_counts[pos]\n    return np.log(counts / counts.sum()).astype(np.float32)\n\n\ndef _prefix_log_probs(drafter, prefix_syms: list[str], max_p: int) -> np.ndarray:\n    pos = len(prefix_syms)\n    if pos >= drafter.L:\n        counts = np.ones(drafter.S, dtype=np.float64)\n        return np.log(counts / counts.sum()).astype(np.float32)\n    for ctx_len in range(min(max_p, pos), -1, -1):\n        ctx = tuple(prefix_syms[-ctx_len:]) if ctx_len > 0 else tuple()\n        key = (pos, ctx)\n        if key in drafter.context_counts:\n            counts = drafter.context_counts[key] + drafter.smoothing\n            return np.log(counts / counts.sum()).astype(np.float32)\n    return _static_log_probs(drafter, pos)\n\n\ndef _row(*, uid, protein_i, length, pipeline, mode, p, k, n_requested, n_used, n_avail,\n         timings, seq, ref, mean_k, acceptance_rate, mean_accepted, n_steps,\n         drafter_accuracy, speedup_runtime, drafter_build_s=0.0,\n         eval_status="ok", error_message=""):\n    drafter_predict_s = timings["drafter_s"]\n    drafter_total_s = "" if drafter_predict_s == "" else drafter_build_s + drafter_predict_s\n    return {\n        "protein_id": uid,\n        "protein_index": protein_i,\n        "length": length,\n        "pipeline": pipeline,\n        "mode": mode,\n        "p": p,\n        "K": k,\n        "n_homologs_requested": n_requested,\n        "n_homologs_used": n_used,\n        "n_homologs_avail": n_avail,\n        "encoder_s": _r(timings["encoder_s"]),\n        "drafter_build_s": _r(drafter_build_s),\n        "drafter_predict_s": _r(drafter_predict_s),\n        "drafter_s": _r(drafter_total_s),\n        "drafter_forward_calls": timings["drafter_forward_calls"],\n        "decoder_s": _r(timings["decoder_s"]),\n        "wall_s": _r(timings["wall_s"]),\n        "peak_vram_gb": _r(timings["peak_vram_gb"]),\n        "speedup_runtime": _r(speedup_runtime) if speedup_runtime != "" else "",\n        "speedup_theoretical_mean_k": _r(mean_k),\n        "acceptance_rate": _r(acceptance_rate),\n        "mean_accepted": _r(mean_accepted),\n        "mean_tokens_per_step": _r(mean_k),\n        "n_steps": n_steps,\n        "drafter_accuracy": _r(drafter_accuracy) if drafter_accuracy != "" else "",\n        "exact_match": seq == ref,\n        "execution_mode": EXECUTION_MODE,\n        "eval_status": eval_status,\n        "error_message": error_message,\n    }\n\n\nFIELDNAMES = [\n    "protein_id", "protein_index", "length", "pipeline", "mode", "p", "K",\n    "n_homologs_requested", "n_homologs_used", "n_homologs_avail",\n    "encoder_s", "drafter_build_s", "drafter_predict_s", "drafter_s",\n    "drafter_forward_calls", "decoder_s", "wall_s", "peak_vram_gb",\n    "speedup_runtime", "speedup_theoretical_mean_k", "acceptance_rate",\n    "mean_accepted", "mean_tokens_per_step", "n_steps", "drafter_accuracy",\n    "exact_match", "execution_mode", "eval_status", "error_message",\n]\n\n\ndef _build_alphabet(loaded):\n    alpha = set()\n    for _, _, projected_rows in loaded:\n        for row in projected_rows:\n            alpha |= {c for c in row if c not in GAP_CHARS}\n    return {c: i for i, c in enumerate(sorted(alpha))}\n\n\ndef _format_aa(seq: str) -> str:\n    return "<AA2fold> " + " ".join(list(seq.upper()))\n\n\ndef _decode_3di(tokenizer, token_ids) -> str:\n    text = tokenizer.decode(token_ids, skip_special_tokens=True)\n    return "".join(text.split()).replace("<AA2fold>", "").lower()\n\n\ndef _clear_cuda():\n    gc.collect()\n    torch.cuda.empty_cache()\n    torch.cuda.ipc_collect()\n\n\ndef _r(value):\n    if value == "":\n        return ""\n    return round(float(value), 6)\n\n\ndef _short_error(exc: BaseException) -> str:\n    return " ".join(str(exc).split())[:500]\n\n\ndef _make_key(protein_id, mode, p, k, n_requested):\n    return (str(protein_id), str(mode), _norm(p), _norm(k), _norm(n_requested), EXECUTION_MODE)\n\n\ndef _completion_key(row):\n    return (\n        str(row["protein_id"]),\n        str(row["mode"]),\n        _norm(row.get("p", "")),\n        _norm(row.get("K", "")),\n        _norm(row.get("n_homologs_requested", "")),\n        str(row.get("execution_mode", EXECUTION_MODE) or EXECUTION_MODE),\n    )\n\n\ndef _has_key(row):\n    return all(field in row for field in COMPLETION_FIELDS)\n\n\ndef _norm(value):\n    text = "" if value is None else str(value).strip()\n    if text.lower() in {"", "nan", "none"}:\n        return ""\n    try:\n        f = float(text)\n    except ValueError:\n        return text\n    return str(int(f)) if f.is_integer() else text\n\n\ndef _existing_baseline_wall(rows, uid):\n    for row in rows:\n        if row.get("protein_id") == uid and row.get("mode") == "benchmark" and row.get("wall_s"):\n            return float(row["wall_s"])\n    return None\n\n\ndef _dedupe(rows):\n    by_key = {}\n    for row in rows:\n        if _has_key(row):\n            by_key[_completion_key(row)] = _normalize_row(row)\n    return [by_key[key] for key in sorted(by_key)]\n\n\ndef _flush_checkpoints(out_path: Path, completed_path: Path, existing_rows, new_rows):\n    all_rows = _dedupe(existing_rows + new_rows)\n    _write_rows(out_path, all_rows, FIELDNAMES)\n    _write_rows(\n        completed_path,\n        [dict(zip(COMPLETION_FIELDS, _completion_key(r))) for r in all_rows],\n        list(COMPLETION_FIELDS),\n    )\n\n\ndef _read_rows(path: Path):\n    if not path.exists() or path.stat().st_size == 0:\n        return []\n    with path.open(newline="") as f:\n        return list(csv.DictReader(f))\n\n\ndef _normalize_row(row):\n    normalized = dict(row)\n    for field in FIELDNAMES:\n        normalized.setdefault(field, "")\n    if normalized.get("eval_status", "") == "":\n        normalized["eval_status"] = "ok"\n    normalized.setdefault("error_message", "")\n    return normalized\n\n\ndef _write_rows(path: Path, rows, fieldnames=None):\n    if not rows:\n        path.write_text("")\n        return\n    keys = fieldnames or list(rows[0].keys())\n    with path.open("w", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows([_normalize_row(row) for row in rows])\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'

helpers = {
    'folding_profile_sweep.py': _PROFILE_SWEEP_SOURCE,
    'folding_hmm_t4_eval.py': _FOLDING_HMM_T4_EVAL_SOURCE,
}
for name, source in helpers.items():
    path = PROST_DIR / name
    if path.exists() and path.read_text() == source:
        print('helper up to date:', path)
    else:
        path.write_text(source)
        print('wrote helper:', path)


helper up to date: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/folding_profile_sweep.py
helper up to date: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/folding_hmm_t4_eval.py


In [4]:
#@title Verify folding_MSA contents
from pathlib import Path

protein_dirs = sorted(p for p in DATA_DIR.iterdir() if p.is_dir())
valid_dirs = [p for p in protein_dirs if (p / 'homologs_projected_to_query_3di.fasta').exists()]
missing_query_aa = [p.name for p in valid_dirs if not (p / 'query_aa.fasta').exists()]
missing_query_3di = [p.name for p in valid_dirs if not (p / 'query_3di.fasta').exists()]

print('Protein directories:', len(protein_dirs))
print('With projected 3Di MSA:', len(valid_dirs))
print('Missing query_aa:', missing_query_aa[:10], 'count=', len(missing_query_aa))
print('Missing query_3di:', missing_query_3di[:10], 'count=', len(missing_query_3di))

if not valid_dirs:
    raise RuntimeError(f'No usable protein folders found in {DATA_DIR}')


Protein directories: 141
With projected 3Di MSA: 141
Missing query_aa: [] count= 0
Missing query_3di: ['A3DH19', 'Q04721'] count= 2


In [5]:
#@title Benchmark configuration
# Protein-first pass on test set 2 only:
# For each protein in test_set_AA_2.fasta, run static first, then p = 1, 2, 4, 8, 16.
# After that protein's rows are checkpointed, the runner moves to the next protein.
STAGE = 'test_set_2_protein_first_p_sweep'
BEST_P = None

K_FIXED = 4
P_SWEEP_VALUES = [1, 2, 4, 8, 16]
N_FOR_P_SWEEP = 16
N_SWEEP_VALUES = [4, 8, 16, 32, 64]
BENCHMARK_FASTA = PROST_DIR / 'prostT5_benchmarks' / 'benchmark_data' / 'test_set_AA_2.fasta'

# First pass: no pure ProstT5 baseline, but include static HMM plus prefix p values.
RUN_BASELINE = False
INCLUDE_STATIC = True
RUN_EACH_P_SEPARATELY = False

# Re-running is safe: completed rows are skipped from folding_hmm_eval_result.csv.
# For a quick T4 smoke test, set PROTEIN_LIMIT = 1 and MAX_CONFIGS = 2.
PROTEIN_LIMIT = None
MAX_CONFIGS = None
FORCE_RECOMPUTE = False
PROGRESS_EVERY = 1

RUN_K_VALUES = [K_FIXED]
RUN_P_VALUES = P_SWEEP_VALUES
RUN_N_VALUES = [N_FOR_P_SWEEP]

print('stage:', STAGE)
print('benchmark FASTA:', BENCHMARK_FASTA)
print('K:', RUN_K_VALUES)
print('p:', RUN_P_VALUES)
print('N homologs:', RUN_N_VALUES)
print('run baseline:', RUN_BASELINE)
print('include static:', INCLUDE_STATIC)
print('protein-first single run:', not RUN_EACH_P_SEPARATELY)
print('protein limit:', PROTEIN_LIMIT)
print('max configs:', MAX_CONFIGS)
print('force recompute:', FORCE_RECOMPUTE)
print('progress every:', PROGRESS_EVERY)
assert BENCHMARK_FASTA.exists(), BENCHMARK_FASTA


stage: test_set_2_protein_first_p_sweep
benchmark FASTA: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/benchmark_data/test_set_AA_2.fasta
K: [4]
p: [1, 2, 4, 8, 16]
N homologs: [16]
run baseline: False
include static: True
protein-first single run: True
protein limit: None
max configs: None
force recompute: False
progress every: 1


In [6]:
#@title Run/resume T4 benchmark
import os, subprocess, sys, shlex, signal, gc


def _cleanup_stale_eval_processes():
    # A failed/aborted eval subprocess can keep the T4 memory allocated and make
    # the next run OOM while loading ProstT5. Kill only old eval subprocesses,
    # never the current notebook kernel.
    try:
        ps = subprocess.run(
            ['ps', '-eo', 'pid=,args='],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            check=False,
        ).stdout.splitlines()
        current_pid = os.getpid()
        killed = []
        for line in ps:
            line = line.strip()
            if not line:
                continue
            pid_text, _, args = line.partition(' ')
            try:
                pid = int(pid_text)
            except ValueError:
                continue
            if pid == current_pid:
                continue
            if 'folding_hmm_t4_eval.py' in args:
                print('Killing stale eval process:', pid, args[:180])
                os.kill(pid, signal.SIGTERM)
                killed.append(pid)
        if killed:
            import time
            time.sleep(3)
            for pid in killed:
                try:
                    os.kill(pid, 0)
                except ProcessLookupError:
                    continue
                print('Force killing stale eval process:', pid)
                os.kill(pid, signal.SIGKILL)
    except Exception as exc:
        print('Stale-process cleanup warning:', repr(exc))

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception as exc:
        print('CUDA cleanup warning:', repr(exc))
    gc.collect()


def _print_gpu_processes(label):
    print(label)
    subprocess.run(['nvidia-smi'], check=False)

cmd = [
    sys.executable,
    '-u',
    str(PROST_DIR / 'folding_hmm_t4_eval.py'),
    '--data', str(DATA_DIR),
    '--out', str(RESULT_DIR),
    '--benchmark-fasta', str(BENCHMARK_FASTA),
    '--progress-every', str(PROGRESS_EVERY),
    '--k-values', ','.join(map(str, RUN_K_VALUES)),
    '--p-values', ','.join(map(str, RUN_P_VALUES)),
    '--n-values', ','.join(map(str, RUN_N_VALUES)),
]
if not RUN_BASELINE:
    cmd += ['--skip-baseline']
if INCLUDE_STATIC:
    cmd += ['--include-static']
if PROTEIN_LIMIT is not None:
    cmd += ['--protein-limit', str(PROTEIN_LIMIT)]
if MAX_CONFIGS is not None:
    cmd += ['--max-configs', str(MAX_CONFIGS)]
if FORCE_RECOMPUTE:
    cmd += ['--force']

_cleanup_stale_eval_processes()
_print_gpu_processes('GPU state before benchmark launch:')

print('Running test set 2 protein-first benchmark:')
print(' '.join(shlex.quote(x) for x in cmd))
print('Order inside each protein: static, then p =', RUN_P_VALUES)
print('Checkpoint CSV:', RESULT_DIR / 'folding_hmm_eval_result.csv')
print('Completed-key CSV:', RESULT_DIR / 'folding_hmm_eval_completed_keys.csv')

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
process = subprocess.Popen(
    cmd,
    cwd=str(PROST_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
if return_code != 0:
    _print_gpu_processes('GPU state after failed benchmark:')
    raise subprocess.CalledProcessError(return_code, cmd)


Killing stale eval process: 143876 /usr/bin/python3 -u /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/folding_hmm_t4_eval.py --data /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_be
GPU state before benchmark launch:
Running test set 2 protein-first benchmark:
/usr/bin/python3 -u /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/folding_hmm_t4_eval.py --data /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA --out /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/results/folding_hmm_results/eval_result --benchmark-fasta /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/benchmark_data/test_set_AA_2.fasta --progress-every 1 --k-values 4 --p-values 1,2,4,8,16 --n-values 16 --skip-baseline --include-static
Order inside each protein: static, then p = [1, 2, 4, 8, 16]
Checkpoint CSV: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/results/folding_hmm_results/

In [ ]:
#@title Run/resume static homolog-count sweep (test set 2, >=64 homologs)
import os, subprocess, sys, shlex, signal, gc
from pathlib import Path

STATIC_HOMOLOG_SWEEP_STAGE = 'test_set_2_static_homolog_count_sweep_min64'
STATIC_H_SWEEP_VALUES = [8, 16, 32, 64]
STATIC_H_SWEEP_K_VALUES = [K_FIXED]
STATIC_H_SWEEP_FASTA = RESULT_DIR / 'test_set_AA_2_min64_homologs.fasta'
STATIC_H_SWEEP_RUN_BASELINE = False
STATIC_H_SWEEP_MIN_HOMOLOGS = 64


def _read_fasta_records(path: Path):
    records = []
    header = None
    seq_parts = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if header is not None:
                records.append((header, ''.join(seq_parts)))
            header = line[1:]
            seq_parts = []
        else:
            seq_parts.append(line)
    if header is not None:
        records.append((header, ''.join(seq_parts)))
    return records


def _count_fasta_records(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for line in path.read_text().splitlines() if line.startswith('>'))


def _write_fasta_records(path: Path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w') as f:
        for header, seq in records:
            f.write(f'>{header}\n')
            for i in range(0, len(seq), 80):
                f.write(seq[i:i + 80] + '\n')



def _ensure_static_only_helper_parser(script_path: Path):
    text = script_path.read_text()
    changed = False
    old_help = 'help="comma-separated prefix p values; default uses the full grid")'
    new_help = 'help="comma-separated prefix p values; default uses the full grid; use none for static-only")'
    if old_help in text:
        text = text.replace(old_help, new_help)
        changed = True

    old_parser = """def _parse_int_list(value: str | None, default: list[int]) -> list[int]:
    if value is None or str(value).strip() == "":
        return list(default)
    return [int(part.strip()) for part in str(value).split(",") if part.strip()]
"""
    new_parser = """def _parse_int_list(value: str | None, default: list[int]) -> list[int]:
    if value is None or str(value).strip() == "":
        return list(default)
    if str(value).strip().lower() in {"none", "static-only", "static_only"}:
        return []
    return [int(part.strip()) for part in str(value).split(",") if part.strip()]
"""
    if old_parser in text:
        text = text.replace(old_parser, new_parser)
        changed = True

    old_min_arg_anchor = '''    parser.add_argument("--n-values", default=None,
                        help="comma-separated homolog-count values; default uses the full grid")
    parser.add_argument("--include-static", action="store_true",
'''
    new_min_arg_anchor = '''    parser.add_argument("--n-values", default=None,
                        help="comma-separated homolog-count values; default uses the full grid")
    parser.add_argument("--min-homologs", type=int, default=0,
                        help="exclude proteins with fewer projected homolog rows before running configs")
    parser.add_argument("--include-static", action="store_true",
'''
    if old_min_arg_anchor in text:
        text = text.replace(old_min_arg_anchor, new_min_arg_anchor)
        changed = True

    old_min_filter_anchor = '''        if missing_ids:
            print("Missing MSA folders for benchmark IDs: " + ", ".join(missing_ids[:30]))
            if len(missing_ids) > 30:
                print(f"... plus {len(missing_ids) - 30} more missing IDs")
    loaded = [load_protein(d) for d in valid_dirs]
'''
    new_min_filter_anchor = '''        if missing_ids:
            print("Missing MSA folders for benchmark IDs: " + ", ".join(missing_ids[:30]))
            if len(missing_ids) > 30:
                print(f"... plus {len(missing_ids) - 30} more missing IDs")
    if args.min_homologs:
        before = len(valid_dirs)
        valid_dirs = [
            p for p in valid_dirs
            if _count_fasta_records(p / "homologs_projected_to_query_3di.fasta") >= args.min_homologs
        ]
        print(
            f"Min-homolog filter: {len(valid_dirs)}/{before} MSA folders have "
            f">= {args.min_homologs} projected homologs"
        )
    loaded = [load_protein(d) for d in valid_dirs]
'''
    if old_min_filter_anchor in text:
        text = text.replace(old_min_filter_anchor, new_min_filter_anchor)
        changed = True

    if 'def _count_fasta_records(path: Path) -> int:' not in text:
        text = text.replace(
            'def _empty_timings():\n',
            'def _count_fasta_records(path: Path) -> int:\n'
            '    if not path.exists():\n'
            '        return 0\n'
            '    return sum(1 for line in path.read_text().splitlines() if line.startswith(">"))\n\n\n'
            'def _empty_timings():\n',
            1,
        )
        changed = True

    if changed:
        script_path.write_text(text)
        print('patched static-only parser in:', script_path)
    else:
        print('static-only parser already present in:', script_path)

def _cleanup_stale_eval_processes_static_sweep():
    try:
        ps = subprocess.run(
            ['ps', '-eo', 'pid=,args='],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            check=False,
        ).stdout.splitlines()
        current_pid = os.getpid()
        killed = []
        for line in ps:
            line = line.strip()
            if not line:
                continue
            pid_text, _, args = line.partition(' ')
            try:
                pid = int(pid_text)
            except ValueError:
                continue
            if pid == current_pid:
                continue
            if 'folding_hmm_t4_eval.py' in args:
                print('Killing stale eval process:', pid, args[:180])
                os.kill(pid, signal.SIGTERM)
                killed.append(pid)
        if killed:
            import time
            time.sleep(3)
            for pid in killed:
                try:
                    os.kill(pid, 0)
                except ProcessLookupError:
                    continue
                print('Force killing stale eval process:', pid)
                os.kill(pid, signal.SIGKILL)
    except Exception as exc:
        print('Stale-process cleanup warning:', repr(exc))

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception as exc:
        print('CUDA cleanup warning:', repr(exc))
    gc.collect()


def _print_gpu_processes_static_sweep(label):
    print(label)
    subprocess.run(['nvidia-smi'], check=False)


source_records = _read_fasta_records(BENCHMARK_FASTA)
eligible_records = []
excluded = []
for header, seq in source_records:
    protein_id = header.split()[0]
    protein_dir = DATA_DIR / protein_id
    homolog_count = _count_fasta_records(protein_dir / 'homologs_projected_to_query_3di.fasta')
    has_query_files = (protein_dir / 'query_aa.fasta').exists() and (protein_dir / 'query_3di.fasta').exists()
    if homolog_count >= 64 and has_query_files:
        eligible_records.append((header, seq))
    else:
        excluded.append((protein_id, homolog_count, has_query_files))

if not eligible_records:
    raise RuntimeError('No test-set-2 proteins have at least 64 projected homologs and query files.')

_write_fasta_records(STATIC_H_SWEEP_FASTA, eligible_records)
print('stage:', STATIC_HOMOLOG_SWEEP_STAGE)
print('source benchmark FASTA:', BENCHMARK_FASTA)
print('eligible FASTA:', STATIC_H_SWEEP_FASTA)
print('eligible proteins:', len(eligible_records), '/', len(source_records))
print('run baseline:', STATIC_H_SWEEP_RUN_BASELINE)
print('min homologs:', STATIC_H_SWEEP_MIN_HOMOLOGS)
print('excluded proteins:', len(excluded))
if excluded:
    print('first excluded:', excluded[:10])

cmd = [
    sys.executable,
    '-u',
    str(PROST_DIR / 'folding_hmm_t4_eval.py'),
    '--data', str(DATA_DIR),
    '--out', str(RESULT_DIR),
    '--benchmark-fasta', str(STATIC_H_SWEEP_FASTA),
    '--progress-every', str(PROGRESS_EVERY),
    '--k-values', ','.join(map(str, STATIC_H_SWEEP_K_VALUES)),
    '--p-values', 'none',
    '--n-values', ','.join(map(str, STATIC_H_SWEEP_VALUES)),
    '--min-homologs', str(STATIC_H_SWEEP_MIN_HOMOLOGS),
    '--include-static',
]
if not STATIC_H_SWEEP_RUN_BASELINE:
    cmd += ['--skip-baseline']
if PROTEIN_LIMIT is not None:
    cmd += ['--protein-limit', str(PROTEIN_LIMIT)]
if MAX_CONFIGS is not None:
    cmd += ['--max-configs', str(MAX_CONFIGS)]
if FORCE_RECOMPUTE:
    cmd += ['--force']

_ensure_static_only_helper_parser(PROST_DIR / 'folding_hmm_t4_eval.py')
_cleanup_stale_eval_processes_static_sweep()
_print_gpu_processes_static_sweep('GPU state before static homolog-count sweep:')

print('Running static homolog-count sweep:')
print(' '.join(shlex.quote(x) for x in cmd))
print('Order inside each protein: static HMM with h =', STATIC_H_SWEEP_VALUES)
print('Checkpoint CSV:', RESULT_DIR / 'folding_hmm_eval_result.csv')
print('Completed-key CSV:', RESULT_DIR / 'folding_hmm_eval_completed_keys.csv')

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
process = subprocess.Popen(
    cmd,
    cwd=str(PROST_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
if return_code != 0:
    _print_gpu_processes_static_sweep('GPU state after failed static homolog-count sweep:')
    raise subprocess.CalledProcessError(return_code, cmd)


stage: test_set_2_static_homolog_count_sweep_min64
source benchmark FASTA: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/benchmark_data/test_set_AA_2.fasta
eligible FASTA: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/results/folding_hmm_results/eval_result/test_set_AA_2_min64_homologs.fasta
eligible proteins: 92 / 100
run baseline: False
min homologs: 64
excluded proteins: 8
first excluded: [('B3H5J1', 13, True), ('Q13117', 21, True), ('B1GT61', 33, True), ('O29316', 18, True), ('Q60307', 0, True), ('A3DH19', 0, False), ('Q55434', 0, False), ('A4YHQ8', 20, True)]
static-only parser already present in: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/folding_hmm_t4_eval.py
GPU state before static homolog-count sweep:
Running static homolog-count sweep:
/usr/bin/python3 -u /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/folding_hmm_t4_eval.py --data /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_b

In [ ]:
#@title Inspect outputs
import pandas as pd

RESULT_CSV = RESULT_DIR / 'folding_hmm_eval_result.csv'
SUMMARY_CSV = RESULT_DIR / 'folding_hmm_eval_run_summary.csv'
COMPLETED_CSV = RESULT_DIR / 'folding_hmm_eval_completed_keys.csv'

print('Result CSV   :', RESULT_CSV)
print('Summary CSV  :', SUMMARY_CSV)
print('Completed CSV:', COMPLETED_CSV)

result_df = pd.read_csv(RESULT_CSV) if RESULT_CSV.exists() and RESULT_CSV.stat().st_size else pd.DataFrame()
summary_df = pd.read_csv(SUMMARY_CSV) if SUMMARY_CSV.exists() and SUMMARY_CSV.stat().st_size else pd.DataFrame()
completed_df = pd.read_csv(COMPLETED_CSV) if COMPLETED_CSV.exists() and COMPLETED_CSV.stat().st_size else pd.DataFrame()

print('rows:', len(result_df))
if len(result_df):
    print('execution modes:', sorted(result_df['execution_mode'].dropna().unique()))
    print('modes:', sorted(result_df['mode'].dropna().unique()))
    print('K:', sorted(pd.to_numeric(result_df['K'], errors='coerce').dropna().astype(int).unique()))
    print('p:', sorted(pd.to_numeric(result_df['p'], errors='coerce').dropna().astype(int).unique()))
    print('N:', sorted(pd.to_numeric(result_df['n_homologs_requested'], errors='coerce').dropna().astype(int).unique()))
    display(result_df.tail(10))

if len(summary_df):
    display(summary_df)
print('completed keys:', len(completed_df))


In [ ]:
#@title Aggregate protein-first p sweep
import pandas as pd

if len(result_df) == 0:
    print('No results yet.')
else:
    MIN_HOMOLOGS_FOR_AGGREGATE = 64
    assisted = result_df[result_df['mode'].isin(['static', 'prefix'])].copy()
    numeric_cols = [
        'K', 'p', 'n_homologs_requested', 'n_homologs_avail',
        'speedup_runtime', 'acceptance_rate', 'peak_vram_gb', 'exact_match', 'wall_s',
    ]
    for col in numeric_cols:
        if col in assisted.columns:
            assisted[col] = pd.to_numeric(assisted[col], errors='coerce')
    if 'eval_status' not in assisted.columns:
        assisted['eval_status'] = 'ok'

    before_filter_rows = len(assisted)
    before_filter_proteins = assisted['protein_id'].nunique()
    assisted = assisted[assisted['n_homologs_avail'] >= MIN_HOMOLOGS_FOR_AGGREGATE].copy()
    print(
        f'Aggregate filter: kept {len(assisted)}/{before_filter_rows} assisted rows and '
        f'{assisted["protein_id"].nunique()}/{before_filter_proteins} proteins with '
        f'n_homologs_avail >= {MIN_HOMOLOGS_FOR_AGGREGATE}.'
    )

    summary = (
        assisted
        .groupby(['mode', 'p', 'K', 'n_homologs_requested'], dropna=False)
        .agg(
            rows=('protein_id', 'count'),
            proteins=('protein_id', 'nunique'),
            ok_rows=('eval_status', lambda s: (s.fillna('ok') == 'ok').sum()),
            oom_rows=('eval_status', lambda s: (s == 'oom').sum()),
            exact_match_rate=('exact_match', 'mean'),
            mean_wall_s=('wall_s', 'mean'),
            mean_speedup_runtime=('speedup_runtime', 'mean'),
            mean_acceptance_rate=('acceptance_rate', 'mean'),
            mean_peak_vram_gb=('peak_vram_gb', 'mean'),
            median_wall_s=('wall_s', 'median'),
            median_speedup_runtime=('speedup_runtime', 'median'),
            median_acceptance_rate=('acceptance_rate', 'median'),
            median_peak_vram_gb=('peak_vram_gb', 'median'),
        )
        .reset_index()
        .sort_values(['mode', 'p', 'K', 'n_homologs_requested'], na_position='first')
    )
    aggregate_path = RESULT_DIR / 'folding_hmm_eval_quick_aggregate.csv'
    mean_aggregate_path = RESULT_DIR / 'folding_hmm_eval_mean_aggregate.csv'
    mean_columns = [
        'mode', 'p', 'K', 'n_homologs_requested', 'rows', 'proteins', 'ok_rows', 'oom_rows',
        'exact_match_rate', 'mean_wall_s', 'mean_speedup_runtime',
        'mean_acceptance_rate', 'mean_peak_vram_gb',
    ]
    summary.to_csv(aggregate_path, index=False)
    summary[mean_columns].to_csv(mean_aggregate_path, index=False)
    print('Saved aggregate:', aggregate_path)
    print('Saved mean aggregate:', mean_aggregate_path)
    display(summary)

    p_stage = summary[
        (summary['mode'] == 'prefix')
        & (summary['K'] == K_FIXED)
        & (summary['n_homologs_requested'] == N_FOR_P_SWEEP)
    ].copy()
    if len(p_stage):
        p_stage = p_stage.sort_values(
            ['exact_match_rate', 'median_wall_s', 'median_acceptance_rate'],
            ascending=[False, True, False],
        )
        suggested_best_p = int(p_stage.iloc[0]['p'])
        print('Suggested BEST_P =', suggested_best_p)
        display(p_stage)
